# Day 2 — Manipulating & Cleaning Data
### Python for Data Analytics

Yesterday we loaded data. Today we make it **trustworthy and useful**: select
the rows we want, fix the messes real data always has, and summarise it.

**Agenda**
1. Select, filter, sort — your Sheets/Excel moves, in pandas
2. Cleaning — names, types, missing values, duplicates, messy text
3. Aggregation & grouping — pivot tables in one line
4. The project — clean a real Kaggle dataset in your own repo (Codespaces)

Run each cell with **Shift + Enter**, in order. 🏋️ **Your turn** cells are for
you to fill in.

---
## Our messy dataset

Real data is never tidy. Here's a small orders table with the problems you meet
all the time. We'll explore it, then fix every issue — the same arc as today's
project.

In [ ]:
import pandas as pd

raw = pd.DataFrame({
    "Order ID":   ["A-1","A-2","A-3","A-4","A-5","A-6","A-7","A-8","A-9","A-10","A-11","A-12","A-3","A-6"],
    "Region":     ["West","East","East","Central","West","Central","East","West","Central","East","West","Central","East","Central"],
    "Category":   ["Furniture"," furniture","OFFICE","Office","Technology"," technology","Furniture","Office","Technology","Office","Furniture","Technology","OFFICE"," technology"],
    "Sales":      ["120.5","90","60","45.0","300","150.75","80","200","75.25","110","95","60","60","150.75"],
    "Units":      [2, None, 1, 3, 1, 2, None, 4, 1, 2, 3, 1, 1, 2],
    "Order Date": ["1/5/2023","2/8/2023","3/2/2023","4/9/2023","5/1/2023","6/3/2023","7/7/2023","8/8/2023","9/9/2023","10/10/2023","11/11/2023","12/12/2023","3/2/2023","6/3/2023"],
})
raw

The first thing you do with *any* new data: look at it. `.info()` shows the
columns, their types, and how many values are filled in.

In [ ]:
raw.info()

Spot the problems already:
- `Sales` is stored as **text** (`object`), not numbers — we can't add it up yet.
- `Units` has **missing** values (fewer non-null than rows).
- `Order Date` is **text**, not a date.
- `Category` is written inconsistently (`OFFICE`, `Office`, ` furniture`).
- Some rows look **duplicated** (`A-3`, `A-6` appear twice).

🏋️ **Your turn.** Print the **shape** of `raw` (rows, columns) and the list of
its column names.

In [ ]:
# TODO: print raw.shape and list(raw.columns)

---
## Block 1 — Select, filter, sort

The everyday spreadsheet moves: pick columns, keep matching rows, order them.

### Selecting columns

One column (a **Series**) uses single brackets. Several columns (a
**DataFrame**) use double brackets.

In [ ]:
raw["Region"]            # one column -> a Series

In [ ]:
raw[["Order ID", "Sales"]]   # several columns -> a DataFrame (note the double [[ ]])

### Filtering rows

Put a condition inside the brackets to keep only the rows where it's `True` —
just like a spreadsheet filter.

In [ ]:
raw[raw["Region"] == "West"]

Combine conditions with `&` (and) / `|` (or). **Each condition needs its own
parentheses.**

In [ ]:
raw[(raw["Region"] == "West") & (raw["Category"] == "Furniture")]

In [ ]:
# Match any of several values with .isin(...)
raw[raw["Region"].isin(["West", "East"])]

🏋️ **Your turn.** Keep only the rows where `Region` is `"Central"`, and show
just the `Order ID` and `Sales` columns.

In [ ]:
# TODO: filter to Region == "Central", then select Order ID and Sales

### Sorting

`sort_values` orders rows. You can sort by one column or several.

In [ ]:
raw.sort_values("Order ID")

In [ ]:
# Sort by Region first, then Order ID within each region.
raw.sort_values(["Region", "Order ID"])

🏋️ **Your turn.** Sort `raw` by `Region`, but in **descending** order.
(Hint: `sort_values("Region", ascending=False)`.)

In [ ]:
# TODO: sort by Region descending

---
## Block 2 — Cleaning

Now we fix the mess, one issue at a time. Each fix below is exactly what you'll
do on the real dataset in the project.

### 1. Tidy the column names

`"Order ID"` is awkward to type (spaces, capitals). Standardise to `order_id`
style — lower case, no spaces.

In [ ]:
raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_")
list(raw.columns)

📝 From here on the columns are `order_id`, `region`, `category`, `sales`,
`units`, `order_date`.

### 2. Fix the types

`to_numeric` turns text into numbers; `to_datetime` turns text into real dates.

In [ ]:
raw["sales"] = pd.to_numeric(raw["sales"])
raw["order_date"] = pd.to_datetime(raw["order_date"], format="%m/%d/%Y")
raw.dtypes

💡 If some values can't be converted (e.g. `"n/a"`), add `errors="coerce"` and
pandas turns the bad ones into `NaN` instead of crashing:

In [ ]:
pd.to_numeric(pd.Series(["10", "20", "n/a"]), errors="coerce")

🏋️ **Your turn.** Convert this price Series to numbers, turning the bad value
into `NaN`: `pd.Series(["1.99", "3.50", "free"])`.

In [ ]:
# TODO: pd.to_numeric(..., errors="coerce")

### 3. Handle missing values

See where the gaps are, then decide: **fill** them or **drop** them.

In [ ]:
raw.isna().sum()

In [ ]:
# Option A: fill the missing Units with 0.
raw["units"] = raw["units"].fillna(0)

# Option B (not used here): raw.dropna(subset=["units"]) would drop those rows.
raw["units"].isna().sum()

🏋️ **Your turn.** Imagine `s = pd.Series([10, None, 30])`. Fill the missing
value with the **mean** of the series. (Hint: `s.fillna(s.mean())`.)

In [ ]:
# TODO: fill the missing value with the mean
s = pd.Series([10, None, 30])

### 4. Remove duplicates

`duplicated()` flags repeated rows; `drop_duplicates()` removes them.

In [ ]:
print("duplicate rows:", raw.duplicated().sum())
raw = raw.drop_duplicates()
print("rows now:", len(raw))

💡 You can also dedupe on *specific* columns: `drop_duplicates(subset=["order_id"])`
keeps one row per order id.

### 5. Clean up messy text

`category` has `"OFFICE"`, `"Office"`, `" furniture"` — the same things written
differently. Strip spaces and standardise the case.

In [ ]:
raw["category"] = raw["category"].str.strip().str.title()
raw["category"].value_counts()

💡 For known one-off fixes, `replace` maps specific values to new ones, e.g.
`df["region"].replace({"W": "West", "E": "East"})`.

🏋️ **Your turn.** Normalise this Series so all three become `"Yes"`:
`pd.Series(["YES", "yes", " Yes "])`. (Hint: `.str.strip().str.title()`.)

In [ ]:
# TODO: normalise to all "Yes"

---
## Block 3 — Aggregation & grouping

Our data is clean — now we summarise it. `value_counts` counts categories;
`groupby` is the pivot table.

In [ ]:
raw["region"].value_counts()

In [ ]:
# Add normalize=True to get proportions instead of counts.
raw["region"].value_counts(normalize=True)

### groupby — the pivot table

Split the data into groups, then compute something per group.

In [ ]:
raw.groupby("region")["sales"].sum()          # total sales per region

In [ ]:
# Several statistics at once.
raw.groupby("region")["sales"].agg(["sum", "mean", "count"])

In [ ]:
# Group by two columns: sales per region AND category.
raw.groupby(["region", "category"])["sales"].sum()

### pivot_table — a 2-D summary

`pivot_table` lays groups out as a grid — regions down the side, categories
across the top — exactly like a spreadsheet pivot table.

In [ ]:
pd.pivot_table(raw, values="sales", index="region", columns="category",
               aggfunc="sum", fill_value=0)

🏋️ **Your turn.** Compute the **average** `sales` per `category`.
(Hint: `groupby("category")` then `.mean()` on `sales`.)

In [ ]:
# TODO: average sales per category

🏋️ **Your turn.** Compute the **total `units`** sold per `region`, and sort the
result from highest to lowest.

In [ ]:
# TODO: total units per region, sorted descending

---
## Block 4 — The project (your turn, for real)

Now you'll do all of this on a **real dataset**, in your **own repository**,
using **GitHub Codespaces** — just like a real data project.

**The dataset:** *Sample - Superstore* on Kaggle (retail orders).
**Setup:** follow the repo's `README.md` to create your repo from the template
and open it in Codespaces, then download the CSV into `data/` (see
`data/README.md`).

Then work through, filling in the `# TODO`s:

| Step | File | What you'll do |
|------|------|----------------|
| 0 | `src/explore.py` | run it — see what's messy (nothing to fill in) |
| 1 | `src/clean.py` | encoding, names, dates, drop columns, duplicates, missing values |
| 2 | `src/transform.py` | new columns, group & summarise, save `summary_*.csv` |
| 3 | `src/questions.py` | answer business questions with code |

Everything you need is in the blocks above — names, types, missing values,
duplicates, text, and `groupby`. Finish by **committing your `output/` files**:
you'll have shipped a small, reproducible cleaning pipeline. 🚀